# Notebook 2: The Collinearity Crash (`Matrix Singularity & VIF`)
**Prepared for:** Assumptions of Least Square Regressions Lecture Suite  
**Author:** Nick Lim

In this notebook, we explore **Assumption 6: No Perfect Multicollinearity**. We will directly examine the linear algebra behind Ordinary Least Squares (OLS):
$$\hat{\beta} = (X'X)^{-1} X'y$$

We will test:
1. **Perfect Collinearity:** What happens when $X_2 = 1.5 \cdot X_1$ exactly? We will see why the determinant $|X'X|$ becomes `0` and `np.linalg.inv` crashes with a `Singular matrix` error.
2. **Imperfect Near-Collinearity:** What if $X_2 = 1.5 \cdot X_1 + \text{tiny noise}$? We will calculate Variance Inflation Factors (VIF) and see how near-collinearity makes $\hat{\beta}$ unstable and highly volatile.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Polyfill for matplotlib 3.9+ compatibility with matplotlib-inline
if not hasattr(matplotlib.rcParams, '_get'):
    matplotlib.rcParams._get = lambda key: matplotlib.rcParams[key]

np.random.seed(42)
sns.set_theme(style="whitegrid")
plt.rcParams["font.size"] = 12

## 1. Perfect Multicollinearity: $X_2 = 1.5 \cdot X_1$

Let's create a dataset where our second predictor variable is an exact 1.5x multiple of our first predictor variable.

In [ ]:
n = 100
x1 = np.random.uniform(1, 10, n)
x2_perfect = 1.5 * x1  # EXACT linear dependency!

y = 5.0 + 2.0 * x1 + 1.0 * x2_perfect + np.random.normal(0, 1.0, n)

# Construct design matrix X (with intercept column of 1s)
X_perfect = sm.add_constant(np.column_stack((x1, x2_perfect)))
print("Design Matrix X shape:", X_perfect.shape)
print("First 3 rows of X:
", X_perfect[:3])

### Calculating $(X'X)$ and its Determinant

To calculate $(X'X)^{-1}$, the matrix $(X'X)$ must have a non-zero determinant (`|X'X| != 0`). Let's calculate the determinant of our $(X'X)$ matrix.

In [ ]:
XtX = X_perfect.T @ X_perfect
det_XtX = np.linalg.det(XtX)

print("Matrix X'X:
", np.round(XtX, 2))
print(f"
Determinant of X'X: {det_XtX:.15f}")

### The Math Explodes: Attempting `np.linalg.inv(X'X)`

Because the determinant is effectively zero ($|X'X| = 0$), attempting to invert the matrix literally requires dividing by zero. Let's see Python throw the exact linear algebra exception!

In [ ]:
try:
    inv_XtX = np.linalg.inv(XtX)
except np.linalg.LinAlgError as e:
    print("⚡ THE COLLINEARITY CRASH CAUGHT! ⚡")
    print("LinAlgError message:", str(e))
    print("
Explanation: Because X2 = 1.5 * X1, the columns are linearly dependent.")
    print("The computer cannot separate the individual effect of X1 from X2 because they move identically!")

## 2. Food for Thought: What if it's *Imperfect* Collinearity?

In real-world data, exact duplicate columns are rare (unless you make a dummy variable trap error). But what happens when variables are **strongly correlated** (`X2 = 1.5 * X1 + tiny_noise`)?

Let's add tiny random noise (`sigma = 0.05`) to `x2` so the determinant is non-zero, and see what happens to our estimated parameters over multiple samples!

In [ ]:
# Adding tiny noise so X2 is 99.9% correlated with X1
x2_near = 1.5 * x1 + np.random.normal(0, 0.05, n)
X_near = sm.add_constant(np.column_stack((x1, x2_near)))

print(f"Correlation between X1 and X2_near: {np.corrcoef(x1, x2_near)[0, 1]:.5f}")

# Calculate Variance Inflation Factors (VIF)
vif_data = pd.DataFrame()
vif_data["Variable"] = ["Intercept", "X1", "X2_near"]
vif_data["VIF"] = [variance_inflation_factor(X_near, i) for i in range(X_near.shape[1])]
print("
=== Variance Inflation Factors (VIF > 10 indicates severe collinearity) ===")
print(vif_data)

### Why High VIF is Dangerous: Coefficient Volatility

Let's run 100 simulations of our near-collinear regression and plot the estimates of $\hat{\beta}_1$ and $\hat{\beta}_2$. Watch how they swing wildly in opposite directions (see-sawing) because the model can't decide which variable gets the credit!

In [ ]:
sim_b1 = []
sim_b2 = []

for _ in range(100):
    y_sim = 5.0 + 2.0 * x1 + 1.0 * x2_near + np.random.normal(0, 1.0, n)
    mod_sim = sm.OLS(y_sim, X_near).fit()
    sim_b1.append(mod_sim.params[1])
    sim_b2.append(mod_sim.params[2])

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(sim_b1, sim_b2, color="#9333ea", alpha=0.7, s=50)
ax.scatter([2.0], [1.0], color="#dc2626", marker="*", s=200, label="True Values ($\beta_1=2.0, \beta_2=1.0$)")
ax.set_title("Instability Under Near-Collinearity: See-Saw Coefficients across Samples")
ax.set_xlabel("Estimated $\hat{\beta}_1$ across Simulations")
ax.set_ylabel("Estimated $\hat{\beta}_2$ across Simulations")
ax.legend()
plt.show()

### Conclusion:
- **Perfect Collinearity ($|X'X| = 0$):** The linear algebra crashes; $(X'X)^{-1}$ does not exist.
- **Imperfect Near-Collinearity ($VIF \gg 10$):** The linear algebra solves, but standard errors explode and estimated coefficients see-saw wildly across different samples.